In [3]:
from djitellopy import Tello
import cv2
import time
import os
from datetime import datetime
 
# --- MODIFIED CATEGORIES ---
# Updated to reflect border security categories as requested
CATEGORIES = ['Drugs', 'Weapons', 'Illegal_Crossing', 'Legal_Crossing']
 
# Create dataset folders for each category 
os.makedirs("dataset/images", exist_ok=True)
os.makedirs("dataset/videos", exist_ok=True)
for category in CATEGORIES:
    os.makedirs(f"dataset/images/{category}", exist_ok=True)
 
# Current category selection 
current_category = CATEGORIES[0] 
category_index = 0
 
tello = Tello()
tello.connect()
print("Battery:", tello.get_battery())
tello.streamon()
time.sleep(3)
 
# Use OpenCV VideoCapture directly 
cap = cv2.VideoCapture('udp://0.0.0.0:11111', cv2.CAP_FFMPEG)
 
if not cap.isOpened():
    print("Error: Could not open video stream") # 
    tello.end()
    exit()
 
# Video recording variables 
recording = False
video_writer = None
frame_width = 960
frame_height = 720
 
# Photo counter for each category 
photo_counters = {cat: 0 for cat in CATEGORIES}
 
print("\n" + "="*60)
print("BORDER SECURITY DATASET COLLECTION") #  - Modified Title
print("="*60)
print("\nControls:")
print("'s' - Save photo to current category")
print("'n' - Next category")
print("'p' - Previous category")
print("'1-4' - Direct category selection") # Modified range to match list size
print("'r' - Start/Stop recording") # [cite: 3]
print("'q' - Quit")
print("\nCategories:")
for i, cat in enumerate(CATEGORIES, 1):
    print(f"  {i}. {cat}") # [cite: 3, 4]
print("="*60)
 
while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        continue
 
    # Resize frame for consistent size [cite: 4]
    frame = cv2.resize(frame, (frame_width, frame_height))
 
    # Display current category (large box at top) [cite: 4, 5]
    cv2.rectangle(frame, (10, 10), (600, 100), (0, 0, 0), -1)
    cv2.rectangle(frame, (10, 10), (600, 100), (0, 0, 255), 3) # Red border for security context
    
    # Updated UI Text for Security Context
    cv2.putText(frame, f"Activity: {current_category.upper()}", 
                (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 3) # [cite: 5]
    cv2.putText(frame, f"Evidence Captured: {photo_counters[current_category]}", 
                (20, 85), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2) # [cite: 5]
 
    # Show recording indicator [cite: 6]
    if recording:
        cv2.circle(frame, (30, 120), 10, (0, 0, 255), -1)
        cv2.putText(frame, "REC", (50, 130), cv2.FONT_HERSHEY_SIMPLEX,
                    0.7, (0, 0, 255), 2)
 
    # Display all category counts (sidebar) [cite: 6, 7]
    y_offset = 120
    cv2.rectangle(frame, (frame_width - 270, 10), (frame_width - 10, 250), (0, 0, 0), -1)
    cv2.putText(frame, "Log Summary:", (frame_width - 260, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
 
    for i, cat in enumerate(CATEGORIES):
        color = (0, 255, 0) if cat == current_category else (150, 150, 150)
        text = f"{i+1}. {cat}: {photo_counters[cat]}"
        cv2.putText(frame, text, (frame_width - 255, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1) # [cite: 8]
        y_offset += 35
 
    # Display battery [cite: 8]
    cv2.putText(frame, f"Battery: {tello.get_battery()}%",
                (10, frame_height - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
 
    cv2.imshow("Tello Border Monitor", frame) # [cite: 10]
 
    # Write frame if recording [cite: 10]
    if recording and video_writer is not None:
        video_writer.write(frame)
 
    key = cv2.waitKey(1) & 0xFF
 
    # Save photo to current category (press 's') [cite: 10, 11]
    if key == ord('s'):
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        photo_counters[current_category] += 1
        # Filename updated to store in new category folders
        filename = f"dataset/images/{current_category}/{current_category}_{photo_counters[current_category]:04d}_{timestamp}.jpg"
        cv2.imwrite(filename, frame)
        print(f"Evidence saved: {filename}")
        print(f"Total {current_category} images: {photo_counters[current_category]}")
 
    # Next category (press 'n') [cite: 11, 12]
    elif key == ord('n'):
        category_index = (category_index + 1) % len(CATEGORIES)
        current_category = CATEGORIES[category_index]
        print(f"\nSwitched to category: {current_category}")
 
    # Previous category (press 'p') [cite: 12]
    elif key == ord('p'):
        category_index = (category_index - 1) % len(CATEGORIES)
        current_category = CATEGORIES[category_index]
        print(f"\nSwitched to category: {current_category}")
 
    # Direct category selection (press '1'-'4') 
    elif key in [ord('1'), ord('2'), ord('3'), ord('4')]:
        idx = int(chr(key)) - 1
        if idx < len(CATEGORIES):
            category_index = idx
            current_category = CATEGORIES[category_index]
            print(f"\nSwitched to category: {current_category}")
 
    # Start/Stop recording (press 'r') [cite: 14]
    elif key == ord('r'):
        if not recording:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"dataset/videos/surveillance_{timestamp}.avi" # Modified filename
            fourcc = cv2.VideoWriter_fourcc(*'XVID')
            video_writer = cv2.VideoWriter(filename, fourcc, 30.0, (frame_width, frame_height))
            recording = True
            print(f"Recording started: {filename}")
        else:
            recording = False
            if video_writer is not None:
                video_writer.release()
                video_writer = None
            print("Recording stopped")
 
    # Quit (press 'q') [cite: 17]
    elif key == ord('q'):
        break
 
# Cleanup [cite: 17]
if recording and video_writer is not None:
    video_writer.release()
cap.release()
cv2.destroyAllWindows()
tello.streamoff()
tello.end()
 
# Print final summary [cite: 17, 18]
print("\n" + "="*60)
print("MISSION SUMMARY")
print("="*60)
print("\nEvidence collected:")
total_images = 0
for cat in CATEGORIES:
    count = photo_counters[cat]
    total_images += count
    print(f"  {cat}: {count} images")
print(f"\nTotal items collected: {total_images}")
print("="*60)

OSError: [WinError 10048] Only one usage of each socket address (protocol/network address/port) is normally permitted

In [3]:
"""
Border Security Detection Model Training Script
This script trains a CNN model to classify different types of border activity
"""

import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import json

# Configuration
IMG_SIZE = 224  # Image size for model input
BATCH_SIZE = 32
EPOCHS = 50
DATASET_PATH = "dataset/images"


CATEGORIES = [
    'weapons',
    'drugs',
    'illegal crossing',
    'legal crossing'
 ]

def create_dataset_structure():
   
    print("Creating dataset folder structure...")
    for category in CATEGORIES:
        os.makedirs(f"{DATASET_PATH}/{category}", exist_ok=True)
    
    for category in CATEGORIES:
        print(f"   - {DATASET_PATH}/{category}/")
    print("\nMove your captured images into the appropriate category folders before training.\n")

def load_and_preprocess_data():
    """
    Load images from organized folders and prepare for training
    """
    print("Loading dataset...")
    data = []
    labels = []
    
    for idx, category in enumerate(CATEGORIES):
        path = os.path.join(DATASET_PATH, category)
        if not os.path.exists(path):
            print(f"Warning: {path} does not exist. Skipping...")
            continue
        
        images = os.listdir(path)
        print(f"Loading {len(images)} images from {category}...")
        
        for img_name in images:
            try:
                img_path = os.path.join(path, img_name)
                img = cv2.imread(img_path)
                if img is None:
                    continue
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                data.append(img)
                labels.append(idx)
            except Exception as e:
                print(f"Error loading {img_name}: {e}")
    
    if len(data) == 0:
        raise ValueError("No images found! Please organize images into category folders.")
    
    data = np.array(data, dtype='float32') / 255.0  # Normalize
    labels = np.array(labels)
    
    print(f"\nTotal images loaded: {len(data)}")
    print(f"Image shape: {data[0].shape}")
    print(f"Number of classes: {len(CATEGORIES)}")
    
    return data, labels

def visualize_samples(data, labels, num_samples=10):
    """
    Visualize sample images from the dataset
    """
    plt.figure(figsize=(15, 6))
    for i in range(min(num_samples, len(data))):
        plt.subplot(2, 5, i + 1)
        plt.imshow(data[i])
        plt.title(CATEGORIES[labels[i]])
        plt.axis('off')
    plt.tight_layout()
    plt.savefig('dataset_samples.png')
    print("Sample images saved as 'dataset_samples.png'")
    plt.close()

def create_cnn_model(num_classes):
    """
    Create a CNN model for border activity classification
    Using MobileNetV2 as base for efficient inference
    """
    base_model = keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    
    # Freeze base model initially
    base_model.trainable = False
    
    # Create model
    model = keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model, base_model

def create_simple_cnn_model(num_classes):

    model = keras.Sequential([
        # First Convolutional Block
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Second Convolutional Block
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Third Convolutional Block
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Dense Layers
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

def train_model(use_transfer_learning=True):
    """
    Main training function
    """
    # Create dataset structure
    create_dataset_structure()
    
    # Load data
    data, labels = load_and_preprocess_data()
    
    # Visualize samples
    visualize_samples(data, labels)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        data, labels, test_size=0.2, random_state=42, stratify=labels
    )
    
    print(f"\nTraining samples: {len(X_train)}")
    print(f"Testing samples: {len(X_test)}")
    
    # Convert labels to categorical
    y_train_cat = keras.utils.to_categorical(y_train, len(CATEGORIES))
    y_test_cat = keras.utils.to_categorical(y_test, len(CATEGORIES))
    
    # Data augmentation
    datagen = ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        horizontal_flip=True,
        zoom_range=0.2,
        shear_range=0.2,
        fill_mode='nearest'
    )
    
    # Create model
    if use_transfer_learning and len(X_train) > 500:
        print("\nUsing Transfer Learning (MobileNetV2)...")
        model, base_model = create_cnn_model(len(CATEGORIES))
    else:
        print("\nUsing Simple CNN (better for small datasets)...")
        model = create_simple_cnn_model(len(CATEGORIES))
        base_model = None
    
    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    print("\nModel Summary:")
    model.summary()
    
    # Callbacks
    callbacks = [
        ModelCheckpoint(
            'best_border_activity_model.keras',
            save_best_only=True,
            monitor='val_accuracy',
            mode='max',
            verbose=1
        ),
        EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1
        )
    ]
    
    # Train model
    print("\nStarting training...")
    history = model.fit(
        datagen.flow(X_train, y_train_cat, batch_size=BATCH_SIZE),
        epochs=EPOCHS,
        validation_data=(X_test, y_test_cat),
        callbacks=callbacks,
        verbose=1
    )
    
    # Fine-tuning (if using transfer learning)
    if base_model is not None and len(X_train) > 1000:
        print("\n" + "="*50)
        print("Fine-tuning the model...")
        print("="*50)
        
        # Unfreeze base model
        base_model.trainable = True
        
        # Compile with lower learning rate
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-5),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        
        # Continue training
        history_fine = model.fit(
            datagen.flow(X_train, y_train_cat, batch_size=BATCH_SIZE),
            epochs=20,
            validation_data=(X_test, y_test_cat),
            callbacks=callbacks,
            verbose=1
        )
    
    # Evaluate model
    print("\n" + "="*50)
    print("Evaluating model...")
    print("="*50)
    
    test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)
    print(f"\nTest Accuracy: {test_acc*100:.2f}%")
    print(f"Test Loss: {test_loss:.4f}")
    
    # Predictions
    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_classes, target_names=CATEGORIES))
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred_classes)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=CATEGORIES, yticklabels=CATEGORIES)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png')
    print("Confusion matrix saved as 'confusion_matrix.png'")
    plt.close()
    
    # Plot training history
    plot_training_history(history)
    
    # Save model and metadata
    model.save('border_activity_detector_model.keras')
    
    # Save model info
    model_info = {
        'categories': CATEGORIES,
        'img_size': IMG_SIZE,
        'test_accuracy': float(test_acc),
        'num_classes': len(CATEGORIES)
    }
    
    with open('model_info.json', 'w') as f:
        json.dump(model_info, f, indent=4)
    
    print("\n" + "="*50)
    print("Training completed!")
    print("="*50)
    print(f"Model saved as: border_activity_detector_model.keras")
    print(f"Best model saved as: best_border_activity_model.keras")
    print(f"Model info saved as: model_info.json")
    
    return model, history

def plot_training_history(history):
    """
    Plot training and validation accuracy/loss
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Accuracy
    ax1.plot(history.history['accuracy'], label='Training Accuracy')
    ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
    ax1.set_title('Model Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True)
    
    # Loss
    ax2.plot(history.history['loss'], label='Training Loss')
    ax2.plot(history.history['val_loss'], label='Validation Loss')
    ax2.set_title('Model Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.savefig('training_history.png')
    print("Training history saved as 'training_history.png'")
    plt.close()

if __name__ == "__main__":
    print("="*60)
    print("BORDER ACTIVITY DETECTION MODEL TRAINING")
    print("="*60)
    
    # Check if organized dataset exists
    organized = all(os.path.exists(f"{DATASET_PATH}/{cat}") for cat in CATEGORIES)
    
    if not organized:
        print("\n Dataset not organized yet!")
        print("\nSteps to prepare your dataset:")
        print("1. Run this script to create folder structure")
        print("2. Organize your captured images into category folders")
        create_dataset_structure()
    else:
        # Train the model
        model, history = train_model(use_transfer_learning=True)

C:\Users\User\anaconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


ILLEGAL ACTIVITY DETECTION MODEL TRAINING
Creating dataset folder structure...
   - dataset/images/weapons/
   - dataset/images/drugs/
   - dataset/images/illegal crossing/
   - dataset/images/legal crossing/

Move your captured images into the appropriate category folders before training.

Loading dataset...
Loading 0 images from weapons...
Loading 0 images from drugs...
Loading 0 images from illegal crossing...
Loading 0 images from legal crossing...


ValueError: No images found! Please organize images into category folders.

In [ ]:
from djitellopy import Tello
import cv2
import time
import numpy as np
from tensorflow import keras
import json
import os

class BorderActivityDetector:
    def __init__(self, model_path='best_border_activity_model.keras', info_path='model_info.json'):
        """
        Initialize the border activity detector
        """
        print("Loading model...")
        
        # Load model
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"Model not found: {model_path}")
        
        self.model = keras.models.load_model(model_path)
        
        # Load model info
        if not os.path.exists(info_path):
            raise FileNotFoundError(f"Model info not found: {info_path}")
        
        with open(info_path, 'r') as f:
            info = json.load(f)
        
        self.categories = info['categories']
        self.img_size = info['img_size']
        self.num_classes = info['num_classes']
        
        print(f"Model loaded successfully!")
        print(f"Categories: {self.categories}")
        print(f"Input size: {self.img_size}x{self.img_size}")
        
    def preprocess_frame(self, frame):
        """
        Preprocess frame for model prediction
        """
        # Resize
        img = cv2.resize(frame, (self.img_size, self.img_size))
        # Convert BGR to RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # Normalize
        img = img.astype('float32') / 255.0
        # Add batch dimension
        img = np.expand_dims(img, axis=0)
        return img
    
    def predict(self, frame):
        """
        Make prediction on a frame
        """
        processed = self.preprocess_frame(frame)
        predictions = self.model.predict(processed, verbose=0)
        class_idx = np.argmax(predictions[0])
        confidence = predictions[0][class_idx]
        
        return self.categories[class_idx], confidence, predictions[0]
    
    def draw_prediction(self, frame, label, confidence, all_probs=None, 
                       show_all_probs=True):
        """
        Draw prediction results on frame
        """
        height, width = frame.shape[:2]
        
        # Choose color based on confidence
        if confidence > 0.7:
            color = (0, 255, 0)  # Green - High confidence
        elif confidence > 0.4:
            color = (0, 255, 255)  # Yellow - Medium confidence
        else:
            color = (0, 0, 255)  # Red - Low confidence
        
        # Draw main prediction
        text = f"{label}: {confidence*100:.1f}%"
        cv2.rectangle(frame, (10, 10), (400, 60), (0, 0, 0), -1)
        cv2.putText(frame, text, (20, 45), cv2.FONT_HERSHEY_SIMPLEX, 
                   1.0, color, 2)
        
        # Draw all probabilities (optional)
        if show_all_probs and all_probs is not None:
            y_offset = 80
            cv2.rectangle(frame, (10, 70), (350, 70 + len(self.categories) * 30), 
                         (0, 0, 0), -1)
            
            # Sort by probability
            sorted_indices = np.argsort(all_probs)[::-1]
            
            for idx in sorted_indices:
                prob = all_probs[idx]
                cat = self.categories[idx]
                bar_width = int(300 * prob)
                
                # Draw probability bar
                cv2.rectangle(frame, (20, y_offset), (20 + bar_width, y_offset + 20),
                            (0, 255, 0), -1)
                
                # Draw text
                text = f"{cat}: {prob*100:.1f}%"
                cv2.putText(frame, text, (25, y_offset + 15), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
                
                y_offset += 30
        
        return frame

def run_tello_detection(show_all_probs=True, save_detections=False):
    """
    Run border activity detection with Tello drone
    """
    # Initialize detector
    try:
        detector = BorderActivityDetector()
    except Exception as e:
        print(f"Error loading model: {e}")
        print("\nPlease train the model first by running: train_border_activity_detector.py")
        return
    
    # Create output folder for detections
    if save_detections:
        os.makedirs("detections", exist_ok=True)
        detection_count = 0
    
    # Connect to Tello
    print("\nConnecting to Tello...")
    tello = Tello()
    tello.connect()
    print(f"Battery: {tello.get_battery()}%")
    
    # Start stream
    tello.streamon()
    time.sleep(3)
    
    # Open video capture
    cap = cv2.VideoCapture('udp://0.0.0.0:11111', cv2.CAP_FFMPEG)
    
    if not cap.isOpened():
        print("Error: Could not open video stream")
        tello.end()
        return
    
    print("\n" + "="*60)
    print("REAL-TIME BORDER ACTIVITY DETECTION ACTIVE")
    print("="*60)
    print("Controls:")
    print("  'q' - Quit")
    print("  's' - Save current detection")
    print("  'p' - Toggle probability display")
    print("="*60)
    
    show_probs = show_all_probs
    fps_list = []
    
    try:
        while True:
            start_time = time.time()
            
            # Read frame
            ret, frame = cap.read()
            if not ret or frame is None:
                continue
            
            # Resize for display
            frame = cv2.resize(frame, (960, 720))
            
            # Make prediction
            label, confidence, all_probs = detector.predict(frame)
            
            # Draw prediction
            frame = detector.draw_prediction(frame, label, confidence, 
                                            all_probs, show_probs)
            
            # Calculate and display FPS
            fps = 1.0 / (time.time() - start_time)
            fps_list.append(fps)
            if len(fps_list) > 30:
                fps_list.pop(0)
            avg_fps = sum(fps_list) / len(fps_list)
            
            # Display FPS and battery
            cv2.putText(frame, f"FPS: {avg_fps:.1f}", (width - 150, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            cv2.putText(frame, f"Battery: {tello.get_battery()}%", 
                       (width - 150, 60),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            
            # Show frame
            cv2.imshow("Tello Border Activity Detector", frame)
            
            # Handle keyboard input
            key = cv2.waitKey(1) & 0xFF
            
            if key == ord('q'):
                break
            elif key == ord('s') and save_detections:
                # Save detection
                filename = f"detections/detection_{detection_count:04d}_{label}.jpg"
                cv2.imwrite(filename, frame)
                print(f"Saved: {filename}")
                detection_count += 1
            elif key == ord('p'):
                # Toggle probability display
                show_probs = not show_probs
                print(f"Probability display: {'ON' if show_probs else 'OFF'}")
            
            # Drone controls
            elif key == ord('t'):
                print("Taking off...")
                tello.takeoff()
            elif key == ord('l'):
                print("Landing...")
                tello.land()
            elif key == ord('w'):
                tello.move_forward(30)
            elif key == ord('s'):
                tello.move_back(30)
            elif key == ord('a'):
                tello.move_left(30)
            elif key == ord('d'):
                tello.move_right(30)
            elif key == ord('e'):
                tello.move_up(30)
            elif key == ord('c'):
                tello.move_down(30)
            elif key == 82:  # Up arrow
                tello.rotate_counter_clockwise(30)
            elif key == 84:  # Down arrow
                tello.rotate_clockwise(30)
    
    except KeyboardInterrupt:
        print("\nStopping...")
    
    finally:
        # Cleanup
        cap.release()
        cv2.destroyAllWindows()
        tello.streamoff()
        tello.end()
        print("\nSession ended.")

def run_webcam_detection(show_all_probs=True):
    """
    Run border activity detection with webcam (for testing without drone)
    """
    # Initialize detector
    try:
        detector = BORDERACTIVITYDetector()
    except Exception as e:
        print(f"Error loading model: {e}")
        print("\nPlease train the model first by running: train_border_detector.py")
        return
    
    # Open webcam
    cap = cv2.VideoCapture(0)
    
    if not cap.isOpened():
        print("Error: Could not open webcam")
        return
    
    print("\n" + "="*60)
    print("WEBCAM BORDER ACTIVITY DETECTION")
    print("="*60)
    print("Controls:")
    print("  'q' - Quit")
    print("  'p' - Toggle probability display")
    print("="*60)
    
    show_probs = show_all_probs
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            # Make prediction
            label, confidence, all_probs = detector.predict(frame)
            
            # Draw prediction
            frame = detector.draw_prediction(frame, label, confidence, 
                                            all_probs, show_probs)
            
            # Show frame
            cv2.imshow("Webcam BORDER ACTIVITY Detector", frame)
            
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
            elif key == ord('p'):
                show_probs = not show_probs
    
    except KeyboardInterrupt:
        print("\nStopping...")
    
    finally:
        cap.release()
        cv2.destroyAllWindows()

if __name__ == "__main__":
    import sys
    
    print("="*60)
    print("BORDER ACTIVITY DETECTION SYSTEM")
    print("="*60)
    
    if len(sys.argv) > 1 and sys.argv[1] == '--webcam':
        # Test with webcam
        run_webcam_detection(show_all_probs=True)
    else:
        # Run with Tello drone
        run_tello_detection(show_all_probs=True, save_detections=True)